# demo - the job the BRIDGE web app runs

The web app's **Run Model** button triggers the Databricks job `MDR_JOB_ID`, which runs this
notebook. It only reads the job parameters and hands them to `mmm.app_job.run_app_job`; all the
logic, and its tests, live there.

**Every run has its own folder**, with its inputs and outputs together, under its BMC and its
**run group** - the modelling period and the modelling type:

```
Secondary Modelling/<bmc_name>/<run_group>/<run_name>/      e.g. .../2025Q1-2025Q4 Secondary/<run_name>/
    run_request.json          written by the app (who, when, the note, reused from)
    Config/ Data/ Prior/      the inputs (Mapping/ and Share/ when used)
    Outputs/                  written by this notebook, also when the run fails
```

A new run starts directly under its group. When someone marks the run the results were reported
from, the web app moves it into `<run_group>/Results Reported/` and the group's other runs into
`<run_group>/Archived/` - never while one of them is running. This notebook never moves anything.

**Job parameters** (define them on the job; all blank by default):

| parameter | meaning |
|---|---|
| `bmc_name` | the BMC folder under `Secondary Modelling/` |
| `run_group` | the period and modelling type folder inside it, e.g. `2025Q1-2025Q4 Secondary` (blank = the run folder sits directly under the BMC, as before the groups) |
| `run_name` | the run's folder inside that |
| `config_file` | the config YAML in the run's `Config/` (blank = this folder's `config.yaml`) |
| `data_file` | the datacube in the run's `Data/` |
| `prior_file` | the feature-prior CSV in the run's `Prior/` |
| `mapping_file` | optional, in the run's `Mapping/` |
| `share_file` | optional, in the run's `Share/` |
| `run_id` | set it to `{{job.run_id}}`: it names the local work folder and is recorded in `run_info.json` |
| `base_path` | blank = `/dbfs/mnt/testuat/Secondary Modelling` (a `/Volumes/...` path works too) |

With `bmc_name`, `run_group` and `run_name` blank, the old shared layout is used: the files are read
from `Secondary Modelling/<Folder>/` and the outputs go to `Outputs/<run_id>`.

Only the input paths, the output folder and the run name are changed; every other setting runs
exactly as uploaded. This folder's `config.yaml` is never edited.

**Running it by hand:** leave every widget blank. It runs this folder's `config.yaml` as it stands,
like `run_real_data.py`, and publishes to `Outputs/manual_<time>`.

The synthetic parameter-recovery check that used to be this notebook's only cell is
`python synthetic_example.py`.

In [ ]:
import json
import os
import sys

# this folder holds the mmm package; make sure it is importable
sys.path.insert(0, os.getcwd())
from mmm.app_job import PARAMS, run_app_job

# Job parameters arrive as widgets. Defining them blank first lets the notebook
# also run by hand. The run id is the job parameter run_id = {{job.run_id}}:
# context.tags() is not available on this compute (it raised a Py4JError).
for name in PARAMS:
    dbutils.widgets.text(name, "")
params = {name: dbutils.widgets.get(name).strip() for name in PARAMS}
print(params)

In [ ]:
# Fits the model and publishes the run folder's Outputs/ (old layout:
# Outputs/<run_id>). A failed run is published first (so its 00_warnings can
# be read) and then re-raised, so the job FAILS
# with the real error - which the web app shows in the run's panel.
info = run_app_job(params)
print(json.dumps(info, indent=2, default=str))

In [ ]:
# the web app reads this through the Jobs API (runs/get-output)
dbutils.notebook.exit(json.dumps(
    {k: info.get(k) for k in ("status", "run_id", "bmc_name", "run_group", "run_name",
                                "codebase", "output_dir", "seconds")},
    default=str))